# Automated Project: Planning, Estimation, and Allocation

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

## Set OpenAI Model

In [2]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

## Loading Tasks and Agents YAML files

In [3]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Create Pydantic Models for Structured Output

In [4]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of tasks with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones")

## Create Crew, Agents and Tasks

In [5]:
# Creating Agents
project_planning_agent = Agent(
  config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
  config=agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
  config=agents_config['resource_allocation_agent']
)

# Creating Tasks
task_breakdown = Task(
  config=tasks_config['task_breakdown'],
  agent=project_planning_agent
)

time_resource_estimation = Task(
  config=tasks_config['time_resource_estimation'],
  agent=estimation_agent
)

resource_allocation = Task(
  config=tasks_config['resource_allocation'],
  agent=resource_allocation_agent,
  output_pydantic=ProjectPlan # This is the structured output we want
)

# Creating Crew
crew = Crew(
  agents=[
    project_planning_agent,
    estimation_agent,
    resource_allocation_agent
  ],
  tasks=[
    task_breakdown,
    time_resource_estimation,
    resource_allocation
  ],
  verbose=True
)

## Crew's Inputs

In [6]:
from IPython.display import display, Markdown

project = 'Website'
industry = 'Technology'
project_objectives = 'Create a website for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

# Format the dictionary as Markdown for a better display in Jupyter Lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}
**Project Requirements:**
{project_requirements}
"""
# Display the formatted output as Markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a website for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)

**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust



## Kicking off the crew

In [7]:
# The given Python dictionary
inputs = {
  'project_type': project,
  'project_objectives': project_objectives,
  'industry': industry,
  'team_members': team_members,
  'project_requirements': project_requirements
}

# Run the crew
result = crew.kickoff(
  inputs=inputs
)

# Agent: The Ultimate Project Planner
## Task: Carefully analyze the project_requirements for the Website project and break them down into individual tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are accounted for:

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build tru



# Agent: Expert Estimation Analyst
## Final Answer: 
### Detailed Estimation Report for Website Project

#### Overview
This report outlines the estimated time, resources, and effort required for each task in the website project while identifying potential risks and uncertainties. The assessment considers historical data, complexity of tasks, and available resources.

### Task Estimates

| **Task**                                           | **Time Estimate** | **Resources Required**       | **Risk/Uncertainty**                          | **Note**                                       |
|----------------------------------------------------|--------------------|-----------------------------|------------------------------------------------|------------------------------------------------|
| 1. Project Kickoff Meeting                        | 1 day               | 1 Project Manager           | Low - Depends on team availability.          | Meeting facilitation and documentation.       |


## Usage Metrics and Costs

Let’s see how much it would cost each time if this crew runs at scale.

In [8]:
import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metrics

Total costs: $0.0028


,total_tokens,prompt_tokens,completion_tokens,successful_requests
0,18373,11070,7303,9


## Result

In [9]:
result.pydantic.dict()

{'tasks': [{'task_name': 'Project Kickoff Meeting',
   'estimated_time_hours': 1.0,
   'required_resources': ['John Doe (Project Manager)']},
  {'task_name': 'Requirement Gathering',
   'estimated_time_hours': 16.0,
   'required_resources': ['Jane Doe', 'Bob Smith']},
  {'task_name': 'Create Wireframes',
   'estimated_time_hours': 40.0,
   'required_resources': ['Bob Smith']},
  {'task_name': 'Develop Responsive Design',
   'estimated_time_hours': 48.0,
   'required_resources': ['Bob Smith', 'Jane Doe']},
  {'task_name': 'UI Development',
   'estimated_time_hours': 104.0,
   'required_resources': ['Jane Doe']},
  {'task_name': 'Navigation System',
   'estimated_time_hours': 16.0,
   'required_resources': ['Jane Doe']},
  {'task_name': 'About Us Page',
   'estimated_time_hours': 16.0,
   'required_resources': ['Jane Doe']},
  {'task_name': 'Services Page',
   'estimated_time_hours': 24.0,
   'required_resources': ['Jane Doe']},
  {'task_name': 'Contact Us Page',
   'estimated_time_hours

## Inspect further

In [10]:
tasks = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(tasks)

# Display the DataFrame as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

,task_name,estimated_time_hours,required_resources
0,Project Kickoff Meeting,1.000000,['John Doe (Project Manager)']
1,Requirement Gathering,16.000000,"['Jane Doe', 'Bob Smith']"
2,Create Wireframes,40.000000,['Bob Smith']
3,Develop Responsive Design,48.000000,"['Bob Smith', 'Jane Doe']"
4,UI Development,104.000000,['Jane Doe']
5,Navigation System,16.000000,['Jane Doe']
6,About Us Page,16.000000,['Jane Doe']
7,Services Page,24.000000,['Jane Doe']
8,Contact Us Page,20.000000,['Jane Doe']
9,Blog Section,40.000000,['Jane Doe']


### Inspecting Milestones

In [11]:
milestones = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestones)

# Display the DataFrame as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

,milestone_name,tasks
0,Project Kickoff,['Project Kickoff Meeting']
1,Requirement Phase,['Requirement Gathering']
2,Design Phase,"['Create Wireframes', 'Develop Responsive Design']"
3,Development Phase,"['UI Development', 'Navigation System', 'About Us Page', 'Services Page', 'Contact Us Page', 'Blog Section', 'Testimonials Section']"
4,SEO and Integration Phase,"['SEO Optimization', 'Social Media Integration']"
5,Quality Assurance Phase,['Quality Assurance Testing']
6,Launch,['Final Adjustments and Launch']
7,Post-Launch Review,['Post-Launch Review']
